# Epsilon-greedy for a stationary k-armed bandit

A stationary k-armed bandit has k actions with fixed, unknown reward distributions. At each step we choose one action and observe only its reward. The objective is to maximize expected cumulative reward.

## Action values and exploration

Greedy action selection uses $A_t = \arg\max_a Q_t(a)$. Epsilon-greedy chooses a random action with probability epsilon and otherwise acts greedily. After the $N_n$-th observation, the incremental sample-average update is $Q_{n+1}=Q_n+(R_n-Q_n)/N_n$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rl_fun.algorithms.bandits.epsilon_greedy import run_epsilon_greedy
from rl_fun.algorithms.bandits.greedy import run_greedy
from rl_fun.algorithms.bandits.random_agent import run_random
from rl_fun.environments.bandit import StationaryBanditEnv
from rl_fun.experiments.compare import aggregate_batch
from rl_fun.experiments.config import ExperimentConfig
from rl_fun.experiments.runner import run_many
from rl_fun.tracking.metrics import MemoryMetricSink

demo_env = StationaryBanditEnv(arms=5, horizon=20)
observation, info = demo_env.reset(seed=7)
observation, reward, terminated, truncated, info = demo_env.step(0)
demo_env.close()
reward, info

In [ ]:
def run_for_notebook(name, seed=42, steps=500):
    env = StationaryBanditEnv(arms=10, horizon=steps)
    env.reset(seed=seed)
    sink = MemoryMetricSink()
    rng = np.random.default_rng(seed)
    if name == "random":
        outcome = run_random(env, steps, rng, sink)
    elif name == "greedy":
        outcome = run_greedy(env, steps, rng, sink)
    else:
        outcome = run_epsilon_greedy(env, steps, rng, sink, epsilon=0.1)
    env.close()
    return outcome, sink


results = {name: run_for_notebook(name) for name in ("random", "greedy", "epsilon_greedy")}

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, (_, sink) in results.items():
    steps = [event.step for event in sink.events]
    axes[0].plot(steps, [event.metrics["cumulative_reward"] for event in sink.events], label=name)
    axes[1].plot(steps, [event.metrics["regret"] for event in sink.events], label=name)
axes[0].set_title("Cumulative reward")
axes[1].set_title("Cumulative regret")
for axis in axes:
    axis.set_xlabel("Step")
    axis.legend()
plt.show()

One seed is a useful illustration, not evidence that one algorithm is better. Random reward draws can make a weak policy look strong (or the reverse), so compare a collection of independently seeded runs.

In [ ]:
batch_config = ExperimentConfig(
    name="notebook-epsilon-greedy",
    algorithm="epsilon_greedy",
    seeds=tuple(range(10)),
    steps=500,
    workers=1,
    output_root="../../runs",
    parameters={"arms": 10, "epsilon": 0.1, "reward_std": 1.0},
)
aggregate_batch(run_many(batch_config))

## Try next

1. Vary epsilon to see the exploration/exploitation trade-off.
2. Change reward noise to see how uncertain observations affect estimates.
3. Change the number of arms to make exploration more or less difficult.